# RankAdaptation — α Sweep on L4 GPU
Tests L8 steering at multiple α values on Qwen2.5-7B-Instruct.

In [ ]:
# Install deps
!pip install -q transformers datasets bitsandbytes accelerate
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Clone repo
!git clone https://github.com/anomalyco/RankAdaptation.git
%cd RankAdaptation

In [ ]:
# Remove old .pt checkpoints from the repo (large files not tracked)
# The TT will be trained locally
!rm -f best_*.pt

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Train standard TT on GSM8K test generations (data generated on-the-fly)
# First collect a small set of trajectories from GSM8K test
import torch, sys, os, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
sys.path.insert(0, '.')
from src.adapters.trajectory_transformer import TrajectoryTransformer

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print('Loading model (4-bit)...')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True,
                                              quantization_config=quant, device_map='cuda')
model.eval()
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok.pad_token = tok.eos_token
print('Loaded')

# Clear GPU cache after model loading
torch.cuda.empty_cache()

In [ ]:
# Collect generation trajectories (memory-safe, CPU storage)
# Uses model.generate() then ONE forward pass for hidden states
device = 'cuda'
N_LAYERS = 28
D_MODEL = 3584
BUDGET = 300  # problems to collect from

ds = load_dataset('openai/gsm8k', 'main', split='test')

all_h, all_v = [], []
for i in range(BUDGET):
    prob = ds[i]
    prompt = f"Q: {prob['question']}\nA:"
    inp = tok(prompt, return_tensors='pt').to(device)
    plen = inp.input_ids.shape[1]

    # Fast generation
    out = model.generate(**inp, max_new_tokens=100, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    gen_len = out.shape[1] - plen
    if gen_len <= 1: continue

    # Single forward for hidden states
    with torch.no_grad():
        fwd = model(out, use_cache=False, output_hidden_states=True)
    hs = fwd.hidden_states

    # Extract generated positions
    h_all = torch.stack([hs[li][0, plen:-1].float() for li in range(N_LAYERS)], dim=1)  # (gen_len, 28, 3584)
    h_next = torch.stack([hs[li][0, plen+1:].float() for li in range(N_LAYERS)], dim=1)
    v = h_next - h_all

    all_h.append(h_all.cpu())
    all_v.append(v.cpu())

    if (i+1) % 50 == 0:
        total = sum(x.shape[0] for x in all_h)
        print(f'  [{i+1}/{BUDGET}] {total} trajs')

h = torch.cat(all_h, dim=0)
v = torch.cat(all_v, dim=0)
print(f'Collected {len(h)} trajectories — keeping on CPU')

# Shuffle + split (stays on CPU)
perm = torch.randperm(len(h))
n_val = len(h) // 10
tr_h, tr_v = h[perm[n_val:]], v[perm[n_val:]]
va_h, va_v = h[perm[:n_val]], v[perm[:n_val]]
del all_h, all_v, h, v

In [ ]:
# Train TT (cache training data on GPU — L4 has 24GB VRAM)
v_mean = tr_v.mean(dim=(0, 1), keepdim=True)
v_std = tr_v.std(dim=(0, 1), keepdim=True) + 1e-8
tr_vn = (tr_v - v_mean) / v_std

# Move ALL training data to GPU once (fits in 24GB)
# 15K × 28 × 3584 × 4 bytes × 2 (h+v) ≈ 12GB
tr_h_gpu = tr_h.to(device)
tr_v_gpu = tr_vn.to(device)
va_h_gpu = va_h.to(device)
va_v_gpu = ((va_v - v_mean) / v_std).to(device)
del tr_h, tr_v, va_h, va_v

tt = TrajectoryTransformer(d_model=768, n_layers=6, n_heads=8, d_ff=3072,
                            n_positions=N_LAYERS, d_input=D_MODEL).to(device)
opt = torch.optim.AdamW(tt.parameters(), lr=3e-4)
bs = 512  # L4 can handle much larger batches
best_r2 = -float('inf')

for ep in range(20):
    tt.train()
    perm = torch.randperm(len(tr_h_gpu))
    for i in range(0, len(tr_h_gpu), bs):
        idx = perm[i:i+bs]
        vp = tt(tr_h_gpu[idx])
        loss = torch.nn.functional.mse_loss(vp, tr_v_gpu[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    
    tt.eval()
    with torch.no_grad():
        vp = tt(va_h_gpu)
        mse = (vp - va_v_gpu).pow(2).mean().item()
        r2 = 1 - mse / max(va_v_gpu.var().item(), 1e-8)
    if r2 > best_r2:
        best_r2 = r2
        torch.save(tt.state_dict(), 'best_gen_tt_7b.pt')
    print(f'ep={ep+1:2d} val_r2={r2:.4f}')

print(f'Best R²={best_r2:.4f} — TT saved')

In [ ]:
!python run_alpha_sweep.py --n-test 100 --layer 8 --alphas -0.5 -0.2 -0.1 0.0 0.05 0.1 0.2 0.5 1.0 2>&1

In [ ]:
# Show results
import json
results = json.load(open('alpha_sweep_L8.json'))
baseline = results['baseline']
print(f"{'Alpha':>6} {'Acc':>8} {'Delta':>8}")
print(f"{'-----':>6} {'---':>8} {'-----':>8}")
for a_str, acc in sorted(results['alphas'].items()):
    a = float(a_str)
    d = acc - baseline if a != 0.0 else 0
    print(f"{a:>6.2f} {100*acc:>7.1f}% {100*d:>+7.1f}pp")

# Download results
from google.colab import files
files.download('alpha_sweep_L8.json')